# Week 9 Study Notebook

# UC San Diego OMDS DSC 207

## Assignment Instructions-
1. You may use any visualization tool - seaborn/matplotlib for a task
2. Sample outputs are provided but your output may vary based on choice of library and parameters.
3. Use the mandatory parameter values wherever specified

# Natural Language Processing with `nltk`

`nltk` is the most popular Python package for Natural Language processing, it provides algorithms for importing, cleaning, pre-processing text data in human language and then apply computational linguistics algorithms like sentiment analysis.

It also includes many easy-to-use datasets in the `nltk.corpus` package, we can download for example the `movie_reviews` package using the `nltk.download` function:

In [1]:
# Uncomment the below line and run this cell if you need to install nltk
#!pip install nltk

In [2]:
#Run this cell for all the imports
import pandas as pd
import numpy as np
import nltk

In [3]:
# Import the sentiment intensity analyzer
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.7 MB/s eta 0:00:00


In [4]:
#Run this cell for later use in tokenization
nltk.download('vader_lexicon')  # for sentiment analysis
nltk.download('punkt_tab')  # for tokenizing

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## 1. Tokenize Text in Words

In [5]:
#Run this cell
romeo_text = """Why then, O brawling love! O loving hate!
O any thing, of nothing first create!
O heavy lightness, serious vanity,
Misshapen chaos of well-seeming forms,
Feather of lead, bright smoke, cold fire, sick health,
Still-waking sleep, that is not what it is!
This love feel I, that feel no love in this."""

The first step in Natural Language processing is generally to split the text into words, this process might appear simple but it is very tedious to handle all corner cases, see for example all the issues with punctuation we have to solve if we just start with a split on whitespace.

1.1 **Split `romeo_text` by spaces and store the resultant list of words in the variable `romeo_tokens`** [0.5 pt]

In [14]:

romeo_tokens = romeo_text.split()

In [15]:
assert type(romeo_tokens) == list, f"romeo_tokens should be a list, got {type(romeo_tokens)}"
assert len(romeo_tokens) == 52, f"romeo_tokens should have 52 words, got {len(romeo_tokens)}"

`nltk` has a sophisticated word tokenizer trained on English named `punkt` which we imported earlier in the notebook.

1.2  **Use the `nltk.word_tokenize(text)` function to properly tokenize `romeo_text` and store the result as `romeo_words`. Print the resultant list.** Compare it to the whitespace splitting we used above and mention the difference. [0.5 pt]

In [16]:

romeo_words = nltk.word_tokenize(romeo_text)

In [17]:
assert type(romeo_words) == list, f"romeo_words should be a list, got {type(romeo_words)}"
assert len(romeo_words) == 68, f"romeo_words should have 68 tokens, got {len(romeo_words)}"

## 2. Build a bag-of-words model

The simplest model for analyzing text is just to think about text as an unordered collection of words (bag-of-words). This can generally allow to infer from the text the category, the topic or the sentiment.

From the bag-of-words model we can build features to be used by a classifier, here we assume that each word is a feature that can either be `True` or `False`.
We implement this in Python as a dictionary where for each word in a sentence we associate `True`.

2.1 **Write a function `build_bag_of_words_features(words)` that returns such a dictionary with {word : True} format given a set of words. Call the function with `romeo_words` and store the resultant dictionary as `romeo_word_dict`.** [1 pt]

In [20]:

def build_bag_of_words_features(words):
    return {word: True for word in words}

In [21]:
romeo_word_dict = build_bag_of_words_features(romeo_words)


In [22]:
# Sanity check
assert type(build_bag_of_words_features(romeo_words)) == dict, "build_bag_of_words_features should return a dictionary"
assert type(romeo_word_dict) == dict, f"romeo_word_dict should be a dictionary, got {type(romeo_word_dict)}"

This is what we wanted, but we notice that also punctuation like "!" and words useless for classification purposes like "of" or "that" are also included.
Those words are named "stopwords" and `nltk` has a convenient corpus we can download:

In [35]:
#Run this cell
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [36]:
#Run this cell
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

Using the Python `string.punctuation` list and the English stopwords we can build better features by filtering out those words that would not help in the classification.

2.2 **Create a list `useless_words` that is a collection of stopwords (from the nltk corpus) in English and the punctuation characters.** [0.5 pt]

In [37]:
from nltk.corpus import stopwords
useless_words = stopwords.words("english") + list(string.punctuation)

In [38]:
assert type(useless_words) == list, f"useless_words should be a list, got {type(useless_words)}"
assert len(useless_words) == 230, f"useless_words should have 230 elements, got {len(useless_words)}"

2.3 **Write a function `build_bag_of_words_features_filtered(words)` that returns a filtered bag of words - a dictionary with only useful words as key and 1 as the value. Call this function with `romeo_words` and store the resultant dictionary as `romeo_useful_word_dict`. (Note: don't convert the words to lowercase before checking if they are a stop word.)** [1 pt]

In [39]:
def build_bag_of_words_features_filtered(words):
    return {word: 1 for word in words if word not in useless_words}

In [40]:
romeo_useful_word_dict = build_bag_of_words_features_filtered(romeo_words)

In [41]:
# Sanity check
assert type(build_bag_of_words_features_filtered(romeo_words)) == dict, "build_bag_of_words_features_filtered should return a dictionary"
assert type(romeo_useful_word_dict) == dict, f"romeo_useful_word_dict should be a dictionary, got {type(romeo_useful_word_dict)}"

## 3. Frequencies of Words

lets explore the movie_reviews dataset before we start the analysis, in this section we will find the most common words and plot their frequency.

3.1. Using the `movie_reviews.words()` (the nltk corpus we imported previously) with no argument we can extract the words from the entire dataset as `all_words` and check that it is about 1.6 million. [0.5 pt]

In [42]:
#we will get the movie review dataset from nltk corpus
nltk.download("movie_reviews")
from nltk.corpus import movie_reviews

all_words = movie_reviews.words()

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.


3.2. Filter out `useless_words` as defined in the previous section(we need to remove the `useless_words` definded in 2.2), and create a new list `filtered_words` this will reduce the length of the dataset by more than a factor of 2. (Hint - Use python list comprehension) [0.5 pt]

In [43]:

filtered_words = filtered_words = [word for word in all_words if word not in useless_words]


In [44]:
assert type(filtered_words) == list, f"filtered_words should be a list, got {type(filtered_words)}"

The `collection` package of the standard library contains a `Counter` class that is handy for counting frequencies of words in our list:

In [45]:
# Run this cell
from collections import Counter
word_counter = Counter(filtered_words)

3.3 Use the [most_common()](https://pythontic.com/containers/counter/most_common) method of `word_counter` to find and store the top 10 most frequently used words from the corpus in `most_common_words`. [0.5 pt]

In [46]:

most_common_words = word_counter.most_common(10)

In [47]:
assert type(most_common_words) == list, f"most_common_words should be a list, got {type(most_common_words)}"
assert len(most_common_words) == 10, f"most_common_words should have 10 elements, got {len(most_common_words)}"

## 4. Sentiment Analysis [2 pt]

Using the sentiment intensity analyzer:
1. Loop over the `list_sentences` (provided below)
2. For each sentence, calculate the polarity scores using `analyzer.polarity_scores(sentence)`
3. Store all the polarity score dictionaries in a list called `sentiment_scores`
4. Print each sentence with its scores

(Hint - refer to lecture notebook)

In [51]:

list_sentences = ["Hello, how are you?", "Today is a nice day", "I don't like the food at the cafe", "This is the worst pizza I have ever had.", "The orange juice is delicious!", "I am late to class."]
analyzer = SentimentIntensityAnalyzer()

sentiment_scores = [analyzer.polarity_scores(sentence) for sentence in list_sentences]

In [53]:
# analyze each text
for text in list_sentences:
    scores = analyzer.polarity_scores(text)
    compound_score = scores['compound']
    sentiment = "Neutral"
    # determine the sentiment category based on compond score
    if compound_score >= 0.05:
        sentiment = "Positive"
    elif compound_score <= -0.05:
        sentiment = "Negative"

    print(f"Text:{text}\nSentiment: {sentiment}, Scores: {scores}\n")


Text:Hello, how are you?
Sentiment: Neutral, Scores: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}

Text:Today is a nice day
Sentiment: Positive, Scores: {'neg': 0.0, 'neu': 0.588, 'pos': 0.412, 'compound': 0.4215}

Text:I don't like the food at the cafe
Sentiment: Negative, Scores: {'neg': 0.232, 'neu': 0.768, 'pos': 0.0, 'compound': -0.2755}

Text:This is the worst pizza I have ever had.
Sentiment: Negative, Scores: {'neg': 0.339, 'neu': 0.661, 'pos': 0.0, 'compound': -0.6249}

Text:The orange juice is delicious!
Sentiment: Positive, Scores: {'neg': 0.0, 'neu': 0.501, 'pos': 0.499, 'compound': 0.6114}

Text:I am late to class.
Sentiment: Neutral, Scores: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}



In [54]:
for s, score in zip(list_sentences, sentiment_scores):
    print(s, score)


Hello, how are you? {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Today is a nice day {'neg': 0.0, 'neu': 0.588, 'pos': 0.412, 'compound': 0.4215}
I don't like the food at the cafe {'neg': 0.232, 'neu': 0.768, 'pos': 0.0, 'compound': -0.2755}
This is the worst pizza I have ever had. {'neg': 0.339, 'neu': 0.661, 'pos': 0.0, 'compound': -0.6249}
The orange juice is delicious! {'neg': 0.0, 'neu': 0.501, 'pos': 0.499, 'compound': 0.6114}
I am late to class. {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}


In [55]:
# Sanity checks
assert type(sentiment_scores) == list, f"sentiment_scores should be a list, got {type(sentiment_scores)}"
assert len(sentiment_scores) == 6, f"sentiment_scores should have 6 elements (one per sentence), got {len(sentiment_scores)}"